## 导入模块与配置

In [ ]:
import sys
from pathlib import Path

# 将 src 目录加入 Python 路径
src_path = Path.cwd() / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 导入自定义模块
from config import OUTPUT_DIR
from data_loader import load_orders
from preprocessing import preprocess_orders
from metrics import compute_user_stats, compute_rfm, compute_category_preference
from repurchase_retention import (
    compute_repeat_purchase_rate,
    compute_monthly_repeat_rate,
    compute_repurchase_cycle,
    plot_repurchase_cycle_distribution,
    compute_retention_matrix
)
from lifetime import compute_ltv_top10, identify_lost_users
from report_exporter import export_user_lifecycle_report

## 加载数据并初步审查

In [ ]:
# 1. 读取数据
users_orders = load_orders()

print("users_orders 前5行:")
print(users_orders.head())
print("\n数据概览:")
print(users_orders.info())

## 预处理：缺失值、异常值、时间特征

In [ ]:
# 2. 数据清洗与特征提取
users_orders = preprocess_orders(users_orders)

print("清洗后数据形状:", users_orders.shape)
print("\n时间特征示例:")
print(users_orders[['order_date', 'order_year', 'order_month', 'order_dayofweek', 'order_quarter']].head())

## 用户基础统计

In [ ]:
# 3. 用户基础统计（第4题）
user_detail = compute_user_stats(users_orders)
print("用户基础统计（前5行）:")
print(user_detail.head())

## RFM 分层

In [ ]:
# 4. RFM 分层（第5题）
rfm_df = compute_rfm(user_detail)
print("RFM 分层结果（前5行）:")
print(rfm_df.head())

## 商品类别偏好

In [ ]:
# 5. 商品类别偏好（第6题）
prefer_df = compute_category_preference(users_orders)
print("用户最偏好商品类别（前5行）:")
print(prefer_df.head())

## 复购率

In [ ]:
# 6. 复购率（第7题）
repeat_rate = compute_repeat_purchase_rate(user_detail)
print(f"整体复购率: {repeat_rate}")

monthly_repeat = compute_monthly_repeat_rate(users_orders)
print("\n月复购率:")
print(monthly_repeat)

## 复购周期分析

In [ ]:
# 7. 复购周期（第8题）
cycle_df = compute_repurchase_cycle(users_orders, user_detail)
global_median = cycle_df['avg'].median()
global_mean = cycle_df['avg'].mean()
print(f"全部用户复购周期中位数：{global_median:.1f} 天")
print(f"全部用户复购周期均值：{global_mean:.1f} 天")

# 绘制分布直方图（与 notebook 一致）
plot_repurchase_cycle_distribution(cycle_df)

## 用户留存曲线

In [ ]:
# 8. 留存曲线（第9题）
retention_pivot = compute_retention_matrix(users_orders)
print("用户留存透视表（前6个月）:")
print(retention_pivot)

## LTV 前十用户

In [ ]:
# 9. LTV 前十用户（第10题）
ltv_top10 = compute_ltv_top10(user_detail)
print("LTV 前十用户:")
print(ltv_top10)

## 流失用户识别

In [ ]:
# 10. 流失用户特征（第11题）
lost_summary = identify_lost_users(user_detail, users_orders)
print("流失用户特征汇总:")
print(lost_summary)

## 导出 Excel 报告

In [ ]:
# 11. 生成 Excel 报告（第12题）
# 构造 sheets 字典，与 notebook 完全一致
sheets = {
    '用户基础统计': user_detail,
    'RFM 分层结果': rfm_df,
    '商品类别偏好': prefer_df,
    '月复购率': monthly_repeat,
    '用户留存透视表': retention_pivot.reset_index(),   # 透视表需 reset_index 才能无索引写入
    '流失用户特征汇总': lost_summary
}

export_user_lifecycle_report(sheets)
print(f"报告已保存至 {OUTPUT_DIR} 目录")